# Tier4 Phase 3B Hybrid Controller (Colab)

Fresh Colab execution notebook for Phase 3B.

Flow:
1. Bootstrap clone + toolchain setup + build.
2. Confirm tool registry.
3. Run `HybridController` (mock agents) for `N` iterations with budgets.
4. Inspect controller bundle + history head.
5. Run controller replay + tool `replay_check` and print PASS.

Notes:
- Uses short `out_root="r"` in `run_cholla` wrapper to avoid path overflows.
- Outputs are intentionally trimmed.


## 1) Bootstrap: Fresh Clone + Dependencies

In [1]:
%%bash
set -euo pipefail
cd /content
rm -rf cholla
git clone --branch dev --single-branch https://github.com/JinchuLi2002/cholla.git
cd cholla
git rev-parse --abbrev-ref HEAD
git rev-parse HEAD


dev
018196872806cd3fedc041dba6506f7d38af7fb9


Cloning into 'cholla'...


In [2]:
%%bash
set -euo pipefail
cd /content/cholla
python3 -m pip install -q --upgrade pip
python3 -m pip install -q h5py pyyaml numpy
python3 -m pip --version


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.9 MB/s eta 0:00:00
pip 26.0.1 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


## 2) Toolchain Setup + Build (Trimmed)

In [3]:
%cd /content/cholla

import glob
import json
import os
import shutil
from pathlib import Path

def _first_match(patterns):
    for pattern in patterns:
        matches = sorted(glob.glob(pattern))
        if matches:
            return Path(matches[0]).resolve()
    return None

def _reset_link(path: Path, target: Path) -> None:
    if path.exists() or path.is_symlink():
        if path.is_symlink() or path.is_file():
            path.unlink()
        else:
            shutil.rmtree(path)
    path.symlink_to(target)

shim_root = Path('/content/cholla/.colab_toolchain')
shim_root.mkdir(parents=True, exist_ok=True)

cuda_header = _first_match([
    '/usr/local/cuda/include/cuda_runtime.h',
    '/usr/local/cuda*/targets/*/include/cuda_runtime.h',
    '/usr/include/cuda_runtime.h',
])
cuda_lib = _first_match([
    '/usr/local/cuda/lib64/libcudart.so',
    '/usr/local/cuda/lib64/libcudart.so.*',
    '/usr/local/cuda*/targets/*/lib/libcudart.so',
    '/usr/local/cuda*/targets/*/lib/libcudart.so.*',
    '/usr/lib/x86_64-linux-gnu/libcudart.so',
    '/usr/lib/x86_64-linux-gnu/libcudart.so.*',
])
if cuda_header is None or cuda_lib is None:
    raise RuntimeError('CUDA toolkit headers/libs not found in this Colab runtime')

cuda_root = shim_root / 'cuda'
(cuda_root / 'include').parent.mkdir(parents=True, exist_ok=True)
_reset_link(cuda_root / 'include', cuda_header.parent)
_reset_link(cuda_root / 'lib64', cuda_lib.parent)

hdf5_header = _first_match([
    '/usr/include/hdf5/serial/hdf5.h',
    '/usr/include/hdf5/openmpi/hdf5.h',
    '/usr/include/hdf5.h',
])
hdf5_lib = _first_match([
    '/usr/lib/x86_64-linux-gnu/hdf5/serial/libhdf5.so',
    '/usr/lib/x86_64-linux-gnu/hdf5/serial/libhdf5_serial.so',
    '/usr/lib/x86_64-linux-gnu/hdf5/openmpi/libhdf5.so',
    '/usr/lib/x86_64-linux-gnu/hdf5/openmpi/libhdf5_openmpi.so',
    '/usr/lib/x86_64-linux-gnu/libhdf5.so',
    '/usr/lib/x86_64-linux-gnu/libhdf5_serial.so',
])
if hdf5_header is None or hdf5_lib is None:
    raise RuntimeError('HDF5 headers/libs not found in this Colab runtime')

hdf5_root = shim_root / 'hdf5'
(hdf5_root / 'include').parent.mkdir(parents=True, exist_ok=True)
_reset_link(hdf5_root / 'include', hdf5_header.parent)

hdf5_lib_dir = hdf5_root / 'lib'
hdf5_lib_dir.mkdir(parents=True, exist_ok=True)
hdf5_link = hdf5_lib_dir / 'libhdf5.so'
if hdf5_link.exists() or hdf5_link.is_symlink():
    hdf5_link.unlink()
hdf5_link.symlink_to(hdf5_lib)

mpi_root = Path('/usr/lib/x86_64-linux-gnu/openmpi')
if not mpi_root.exists():
    raise RuntimeError(f'MPI root missing: {mpi_root}')

env_payload = {
    'CUDA_ROOT': str(cuda_root),
    'HDF5_ROOT': str(hdf5_root),
    'MPI_ROOT': str(mpi_root),
}

artifacts = Path('/content/cholla/artifacts')
artifacts.mkdir(parents=True, exist_ok=True)
(artifacts / 'env_colab.json').write_text(json.dumps(env_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(env_payload, sort_keys=True))


/content/cholla
{"CUDA_ROOT": "/content/cholla/.colab_toolchain/cuda", "HDF5_ROOT": "/content/cholla/.colab_toolchain/hdf5", "MPI_ROOT": "/usr/lib/x86_64-linux-gnu/openmpi"}


In [4]:
%%bash
set -euo pipefail
cd /content/cholla
eval "$(python3 - <<'PY'
import json
import shlex
from pathlib import Path
payload = json.loads(Path('artifacts/env_colab.json').read_text(encoding='utf-8'))
for key in ('CUDA_ROOT', 'HDF5_ROOT', 'MPI_ROOT'):
    print(f'export {key}={shlex.quote(payload[key])}')
PY
)"
build_log=/tmp/cholla_phase3b_build.log
env CHOLLA_MACHINE=github CUDA_ROOT="$CUDA_ROOT" HDF5_ROOT="$HDF5_ROOT" MPI_ROOT="$MPI_ROOT" make TYPE=cosmology -j2 >"$build_log" 2>&1
tail -n 25 "$build_log"
ls -lh bin/cholla.cosmology.github


      Real const chi_1 = C * (del_m_i.density + d_6) + D * d_6;
                 ^

nvcc -g -O3 -std=c++17 -DMPI_CHOLLA -DPRECISION=2 -DHLLC -DSIMPLE -DPPMP -DDENSITY_FLOOR -DTEMPERATURE_FLOOR -DOUTPUT -DHDF5  -DGRAVITY -DPARIS -DGRAVITY_GPU -DGRAVITY_5_POINTS_GRADIENT -DPARALLEL_OMP -DN_OMP_THREADS=7  -DPARTICLES -DPARTICLES_GPU -DPARTICLE_IDS -DSINGLE_PARTICLE_MASS -DPARALLEL_OMP -DN_OMP_THREADS=7 -DCOSMOLOGY -DAVERAGE_SLOW_CELLS -DDE -DPRINT_INITIAL_STATS -DN_OUTPUT_COMPLETE=1 -DPARIS_5PT -DGIT_HASH='"018196872806cd3fedc041dba6506f7d38af7fb9"' -DMACRO_FLAGS='"-DMPI_CHOLLA -DPRECISION=2 -DHLLC -DSIMPLE -DPPMP -DDENSITY_FLOOR -DTEMPERATURE_FLOOR -DOUTPUT -DHDF5  -DGRAVITY -DPARIS -DGRAVITY_GPU -DGRAVITY_5_POINTS_GRADIENT -DPARALLEL_OMP -DN_OMP_THREADS=7  -DPARTICLES -DPARTICLES_GPU -DPARTICLE_IDS -DSINGLE_PARTICLE_MASS -DPARALLEL_OMP -DN_OMP_THREADS=7 -DCOSMOLOGY -DAVERAGE_SLOW_CELLS -DDE -DPRINT_INITIAL_STATS -DN_OUTPUT_COMPLETE=1 -DPARIS_5PT -DGIT_HASH='"018196872806cd3fedc041dba650

## 3) Confirm Tool Registry

In [5]:
%cd /content/cholla

import json
from agent.tools.registry import TOOLS

required_tools = ['validate_params', 'run_cholla', 'compute_metric', 'replay_check']
payload = {
    'tool_count': len(TOOLS),
    'required_present': {name: (name in TOOLS) for name in required_tools},
    'tool_names_head': sorted(TOOLS.keys())[:8],
}
print(json.dumps(payload, sort_keys=True))


/content/cholla
{"required_present": {"compute_metric": true, "replay_check": true, "run_cholla": true, "validate_params": true}, "tool_count": 6, "tool_names_head": ["compute_metric", "compute_metric_v3", "replay_check", "replay_verify", "run_cholla", "validate_params"]}


## 4) Create Linked Experiment ID for `replay_check` (1 Iter, Trimmed)

In [6]:
%cd /content/cholla

import json
import subprocess
from pathlib import Path

repo_root = Path('/content/cholla')
cmd = [
    'python3',
    'agent/run_agent.py',
    '--seed',
    '123',
    '--iters',
    '1',
    '--max-total-runs',
    '1',
    '--max-failures',
    '1',
    '--agent-run-id',
    'phase3b_linked',
]
proc = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True, check=False)

summary_line = ''
for line in reversed(proc.stdout.splitlines()):
    if '"summary"' in line:
        summary_line = line.strip()
        break

linked_experiment_id = ''
if summary_line:
    payload = json.loads(summary_line)
    summary = payload.get('summary', {})
    if isinstance(summary, dict):
        value = summary.get('experiment_id')
        if isinstance(value, str):
            linked_experiment_id = value

out_payload = {
    'returncode': proc.returncode,
    'experiment_id': linked_experiment_id,
    'stdout_tail': proc.stdout.splitlines()[-5:],
    'stderr_tail': proc.stderr.splitlines()[-5:],
}

out_path = repo_root / 'artifacts' / 'phase3b_linked_experiment.json'
out_path.write_text(json.dumps(out_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps({'linked_experiment_path': str(out_path), 'experiment_id': linked_experiment_id, 'returncode': proc.returncode}, sort_keys=True))
if not linked_experiment_id:
    raise RuntimeError('Failed to derive linked experiment_id for replay_check')


/content/cholla
{"experiment_id": "exp_123_cc9c6920a808", "linked_experiment_path": "/content/cholla/artifacts/phase3b_linked_experiment.json", "returncode": 0}


## 5) Run HybridController (Mock Agents, Tool-Only, Budgeted)

This run enforces short `out_root='r'` by wrapping `run_cholla` in the tool registry.


In [7]:
%cd /content/cholla

import json
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

from agent.controller.agents.planner import MockPlannerAgent
from agent.controller.agents.summarizer import MockSummarizerAgent
from agent.controller.controller import HybridController
from agent.tools.registry import TOOLS
from agent.tools.run_cholla import run_cholla as run_cholla_tool

repo_root = Path('/content/cholla').resolve()
linked_payload = json.loads((repo_root / 'artifacts' / 'phase3b_linked_experiment.json').read_text(encoding='utf-8'))
linked_experiment_id = linked_payload.get('experiment_id', '')
if not isinstance(linked_experiment_id, str) or not linked_experiment_id:
    raise RuntimeError('linked experiment id missing')

def run_cholla_short(payload):
    patched = dict(payload)
    patched['out_root'] = 'r'
    return run_cholla_tool(patched)

tool_registry = deepcopy(TOOLS)
run_desc = dict(tool_registry['run_cholla'])
run_desc['callable'] = run_cholla_short
tool_registry['run_cholla'] = run_desc

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
experiment_id = f'ctrl_phase3b_{stamp}'
controller = HybridController(
    repo_root=repo_root,
    summarizer=MockSummarizerAgent(lookback_k=5),
    planner=MockPlannerAgent(seed=123),
    tool_registry=tool_registry,
    experiment_id=experiment_id,
    controller_run_id=f'{experiment_id}_run',
    max_iterations=2,
    max_tool_calls=12,
    walltime_budget_sec=900,
)

result = controller.run(initial_state={'linked_experiment_id': linked_experiment_id})
result_path = repo_root / 'artifacts' / 'phase3b_controller_result.json'
result_path.write_text(json.dumps(result, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print(json.dumps({
    'result_path': str(result_path),
    'experiment_id': result['experiment_id'],
    'history_path': result['history_path'],
    'experiment_bundle_path': result['experiment_bundle_path'],
    'forced_out_root': 'r',
}, sort_keys=True))


/content/cholla


ModuleNotFoundError: No module named 'agent.controller'

## 6) Inspect Bundle Tree + History Head

In [ ]:
%cd /content/cholla

import json
from pathlib import Path

repo_root = Path('/content/cholla').resolve()
result = json.loads((repo_root / 'artifacts' / 'phase3b_controller_result.json').read_text(encoding='utf-8'))
bundle = Path(result['experiment_bundle_path'])

print(f'bundle={bundle}')
shown = 0
for path in sorted(bundle.rglob('*')):
    rel = path.relative_to(bundle)
    if len(rel.parts) > 4:
        continue
    suffix = '/' if path.is_dir() else ''
    print(rel.as_posix() + suffix)
    shown += 1
    if shown >= 80:
        print('... trimmed ...')
        break


In [ ]:
%cd /content/cholla

import json
from pathlib import Path

repo_root = Path('/content/cholla').resolve()
result = json.loads((repo_root / 'artifacts' / 'phase3b_controller_result.json').read_text(encoding='utf-8'))
history_path = Path(result['history_path'])
print(f'history={history_path}')
for idx, line in enumerate(history_path.read_text(encoding='utf-8').splitlines()[:5], start=1):
    print(f'{idx:02d}: {line[:300]}')


## 7) Replay Checks + PASS Gate

In [ ]:
%cd /content/cholla

import json
from copy import deepcopy
from pathlib import Path

from agent.controller.agents.planner import MockPlannerAgent
from agent.controller.agents.summarizer import MockSummarizerAgent
from agent.controller.controller import HybridController
from agent.tools.registry import TOOLS
from agent.tools.replay_check import replay_check
from agent.tools.run_cholla import run_cholla as run_cholla_tool

repo_root = Path('/content/cholla').resolve()
result = json.loads((repo_root / 'artifacts' / 'phase3b_controller_result.json').read_text(encoding='utf-8'))
linked_payload = json.loads((repo_root / 'artifacts' / 'phase3b_linked_experiment.json').read_text(encoding='utf-8'))
linked_experiment_id = linked_payload.get('experiment_id', '')
if not isinstance(linked_experiment_id, str) or not linked_experiment_id:
    raise RuntimeError('linked experiment id missing for replay_check')

def run_cholla_short(payload):
    patched = dict(payload)
    patched['out_root'] = 'r'
    return run_cholla_tool(patched)

tool_registry = deepcopy(TOOLS)
run_desc = dict(tool_registry['run_cholla'])
run_desc['callable'] = run_cholla_short
tool_registry['run_cholla'] = run_desc

controller = HybridController(
    repo_root=repo_root,
    summarizer=MockSummarizerAgent(lookback_k=5),
    planner=MockPlannerAgent(seed=123),
    tool_registry=tool_registry,
    experiment_id=result['experiment_id'],
    controller_run_id=result['controller_run_id'],
    max_iterations=2,
    max_tool_calls=12,
    walltime_budget_sec=900,
)

controller_replay = controller.replay(result['experiment_id'])
tool_replay = replay_check({'experiment_id': linked_experiment_id})

print(json.dumps({
    'controller_replay': controller_replay,
    'tool_replay_check': tool_replay,
}, sort_keys=True))

ok = controller_replay.get('status') == 'ok' and tool_replay.get('status') == 'ok'
print('PHASE3B_PASS' if ok else 'PHASE3B_FAIL')
if not ok:
    raise RuntimeError('Phase 3B replay checks did not pass')
